# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 2 — APIs, prompting y configuración

ITBA Educación Ejecutiva

Del modelo a una aplicación de software: hoy dejamos de ver el LLM "desde afuera" (Clase 1) y empezamos a programar contra él.

### Cómo se usa este notebook

- **Durante la clase en vivo**: el docente corre cada celda y explica el resultado — no hace falta que vengas con el entorno funcionando de antemano para seguir la clase, aunque sí para el laboratorio.
- **Celdas 🏠 Practicá**: son ejercicios cortos para resolver **después de la clase**, no en vivo. Van a la par de cada sección para consolidar el concepto recién visto.
- **Sección 9 (Laboratorio)**: es la tarea principal de la semana — el checkpoint del proyecto integrador. Se entrega/completa antes de la Clase 3, no durante esta clase.

## Objetivo de la clase

Al final de la clase vas a poder:
- Configurar un entorno de desarrollo Python con dependencias y credenciales manejadas de forma segura.
- Consumir un modelo de lenguaje vía API, distinguiendo mensajes de sistema y de usuario.
- Aplicar zero-shot y few-shot prompting, y delimitar instrucciones/contexto en el prompt.
- Controlar parámetros de generación (temperatura, top_p, max_tokens, stop sequences).
- Manejar errores básicos de la API (rate limits, validación, timeouts).
- Explicar la diferencia entre llamadas síncronas y asíncronas, y cuándo conviene cada una.

**Checkpoint del proyecto integrador de hoy**: cliente funcional para consumir un modelo generativo (ver `proyecto-integrador/README.md`, checkpoint Clase 2).

## Agenda

1. Configuración del entorno (dependencias, variables de entorno, credenciales)
2. Tu proveedor: referencia rápida (Bedrock, Anthropic, OpenAI, Google, Ollama)
3. Primer cliente: consumo de la API
4. Mensajes de sistema y usuario
5. Zero-shot vs. few-shot prompting
6. Delimitación de instrucciones y contexto
7. Parámetros de generación
8. Manejo básico de errores
9. Síncrono vs. asíncrono
10. Laboratorio: cliente básico reutilizable

## 1. Configuración del entorno

Antes de escribir código, el entorno tiene que estar reproducible: cualquier otra persona (o vos en 3 meses) debería poder clonar el repo y correr esto sin adivinar nada.

**Entorno virtual** (aísla las dependencias del proyecto del resto del sistema):

```bash
python3 -m venv .venv
source .venv/bin/activate   # en Windows: .venv\Scripts\activate
```

**Dependencias** — instalamos lo que vamos a usar hoy:

```bash
pip install boto3 python-dotenv
```

**Credenciales** — nunca se hardcodean. Se cargan desde variables de entorno vía un archivo `.env` local (gitignoreado). Ver `.env.example` en la raíz del repo — completá la sección del proveedor que vayas a usar (por defecto, AWS Bedrock, provisto por la cátedra).

**Dependencias con versión fija** — `pip install` sin versión instala "lo último de hoy", que puede romper mañana. En un proyecto real se fijan versiones (`requirements.txt` o `pyproject.toml`) para que el entorno sea reproducible bit a bit:

```bash
pip freeze > requirements.txt   # después de instalar lo que necesitás
pip install -r requirements.txt # en cualquier otra máquina, instala exactamente lo mismo
```

In [ ]:
# Si estás corriendo esta celda en el mismo entorno donde ya instalaste las dependencias, podés omitir esto.
# Descomentar si hace falta instalar en el kernel actual:
# %pip install boto3 python-dotenv

**🏠 Practicá (tarea, después de la clase)**: corré `pip freeze` en tu terminal (con el venv activado) y mirá cuántos paquetes aparecen instalados además de los dos que pediste explícitamente — son las dependencias transitivas de `boto3`/`python-dotenv`. Guardá el resultado en un `requirements.txt` dentro de tu carpeta de trabajo del TP.

In [2]:
import os
from dotenv import load_dotenv

# Carga las variables de entorno desde .env (buscando hacia arriba desde el cwd)
load_dotenv()

provider = os.environ.get("LLM_PROVIDER", "bedrock")
print(f"Proveedor configurado: {provider}")

# Chequeo de credenciales según el proveedor elegido (ver .env.example)
credenciales_esperadas = {
    "bedrock": "AWS_BEARER_TOKEN_BEDROCK",
    "anthropic": "ANTHROPIC_API_KEY",
    "openai": "OPENAI_API_KEY",
    "google": "GOOGLE_API_KEY",
    "ollama": None,  # Ollama corre local, no necesita API key
}
var_esperada = credenciales_esperadas.get(provider)
if var_esperada:
    assert os.environ.get(var_esperada), (
        f"Falta {var_esperada} en el .env para el proveedor '{provider}' — "
        f"copiá .env.example a .env y completá esa sección."
    )
print("Credenciales OK." if var_esperada else "Ollama: no requiere API key, solo que esté corriendo localmente.")

Proveedor configurado: bedrock   # valores posibles: bedrock | anthropic | openai | google | ollama
Ollama: no requiere API key, solo que esté corriendo localmente.


## 2. Tu proveedor: referencia rápida

**El curso es agnóstico de proveedor** — no todos tienen acceso a AWS Bedrock, así que elegí el que tengas disponible (Anthropic, OpenAI, Google o un modelo local con Ollama). En la demo de hoy el docente usa Bedrock (es el acceso que provee la cátedra), pero el laboratorio de la sección 10 lo resolvés con **el que vos tengas**. Guardá esta tabla — la vas a usar en el laboratorio.

Instalación (agregá solo la línea del proveedor que vayas a usar):

```bash
pip install boto3            # AWS Bedrock
pip install anthropic        # Anthropic directo
pip install openai           # OpenAI
pip install google-genai     # Google Gemini
pip install ollama           # Modelos locales
```

In [ ]:
# --- AWS Bedrock ---
import boto3

def llamar_bedrock(prompt, system=None, temperature=0.7, max_tokens=500):
    client = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
    kwargs = {"modelId": os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6"),
              "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens}}
    if system:
        kwargs["system"] = [{"text": system}]
    resp = client.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"]

In [ ]:
# --- Anthropic directo ---
import anthropic

def llamar_anthropic(prompt, system=None, temperature=0.7, max_tokens=500):
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    kwargs = {"model": os.environ.get("ANTHROPIC_MODEL_ID", "claude-sonnet-4-6"),
              "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    # Nota: en algunos modelos recientes de Anthropic, `temperature` no es configurable
    # (devuelve error 400 si se manda con un valor distinto del default) — omitirlo si falla.
    resp = client.messages.create(**kwargs)
    return resp.content[0].text

In [ ]:
# --- OpenAI ---
import openai
from openai import OpenAI

def llamar_openai(prompt, system=None, temperature=0.7, max_tokens=500):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    try:
        resp = client.chat.completions.create(
            model=os.environ.get("OPENAI_MODEL_ID", "gpt-4o"),
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content
    except openai.RateLimitError:
        raise
    except openai.APIStatusError as e:
        raise RuntimeError(f"Error de la API de OpenAI: {e.status_code}") from e

In [ ]:
# --- Google Gemini ---
from google import genai
from google.genai import types

def llamar_google(prompt, system=None, temperature=0.7, max_tokens=500):
    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        max_output_tokens=max_tokens,
    )
    resp = client.models.generate_content(
        model=os.environ.get("GOOGLE_MODEL_ID", "gemini-2.0-flash"),
        contents=prompt,
        config=config,
    )
    return resp.text

In [ ]:
# --- Ollama (modelo local) ---
import ollama

def llamar_ollama(prompt, system=None, temperature=0.7, max_tokens=500):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = ollama.chat(
        model=os.environ.get("OLLAMA_MODEL_ID", "llama3.1"),
        messages=messages,
        options={"temperature": temperature, "num_predict": max_tokens},
    )
    return resp["message"]["content"]

Notá que las 5 funciones hacen exactamente lo mismo (mandar un prompt + system opcional, devolver texto) pero cada una con su propia forma de llamar y de leer la respuesta — es justamente el problema que la Clase 3 resuelve con una interfaz común.

## 3. Primer cliente: consumo de la API

El resto de esta clase usa **AWS Bedrock** para la demo en vivo (es el acceso que provee la cátedra), con la función `llamar_bedrock` de arriba envuelta en una variable `bedrock` + `model_id` para que el código sea más corto de leer. Si estás siguiendo con otro proveedor, reemplazá las llamadas por la función equivalente de la sección anterior — la lógica (prompting, parámetros, errores) es la misma en todos.

In [13]:
import boto3

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

response = bedrock.converse(
    modelId="global.anthropic.claude-sonnet-4-6",
    messages=[
        {"role": "user", "content": [{"text": "¿Qué es Retrieval-Augmented Generation, en una oración?"}]}
    ],
)

texto = response["output"]["message"]["content"][0]["text"]
print(texto)
print()
print("Uso de tokens:", response["usage"])
print("Motivo de fin:", response["stopReason"])

AccessDeniedException: An error occurred (AccessDeniedException) when calling the Converse operation: User: arn:aws:iam::032474884952:user/Curso_escuela_de_innovacion is not authorized to perform: bedrock:InvokeModel on resource: arn:aws:bedrock:us-east-1:032474884952:inference-profile/global.anthropic.claude-sonnet-4-6 because no identity-based policy allows the bedrock:InvokeModel action

### Anatomía de la respuesta

El dict que devuelve `converse()` siempre tiene esta forma (independiente del modelo que uses en Bedrock):

```
{
  "output": {"message": {"role": "assistant", "content": [{"text": "..."}]}},
  "stopReason": "end_turn" | "max_tokens" | "stop_sequence" | ...,
  "usage": {"inputTokens": N, "outputTokens": N, "totalTokens": N},
  "metrics": {"latencyMs": N}
}
```

`usage` es clave para estimar costo (Clase 3: registro de tokens y costos), y `stopReason` te dice *por qué* terminó — no es lo mismo que haya terminado porque el modelo llegó a un final natural (`end_turn`) a que se haya cortado por `max_tokens`.

In [ ]:
print("Latencia (ms):", response["metrics"]["latencyMs"])
print("Tokens de entrada:", response["usage"]["inputTokens"])
print("Tokens de salida:", response["usage"]["outputTokens"])

**🏠 Practicá (tarea, después de la clase)**: cambiá la pregunta de la celda anterior por una sobre algún tema de la Clase 1 (ej. "¿qué es BPE?") y volvé a correr. Compará `inputTokens` de las dos preguntas — ¿por qué no da lo mismo aunque las dos sean cortas?

## 4. Mensajes de sistema y usuario

- **System prompt**: instrucciones/contexto que definen el rol o comportamiento del modelo para toda la conversación. Se manda una sola vez, aparte de los mensajes.
- **User / assistant messages**: la conversación en sí — turnos alternados. Para mantener contexto entre llamadas, hay que reenviar el historial completo en cada request (el modelo no tiene memoria entre llamadas).

In [ ]:
system_prompt = [
    {"text": "Sos un asistente técnico para una certificación de desarrollo de agentes de IA. "
             "Respondé en español, en un máximo de 3 oraciones, en tono directo y sin relleno."}
]

messages = [
    {"role": "user", "content": [{"text": "¿Para qué sirve la ventana de contexto de un LLM?"}]}
]

response = bedrock.converse(
    modelId=model_id,
    system=system_prompt,
    messages=messages,
)

print(response["output"]["message"]["content"][0]["text"])

### Multi-turno: el modelo no tiene memoria propia

Cada llamada a `converse()` es independiente — si querés que el modelo "recuerde" lo que dijo antes, tenés que reenviar la conversación completa, agregando la respuesta anterior como mensaje `assistant`.

In [ ]:
# Turno 1
messages = [{"role": "user", "content": [{"text": "Estoy comparando Claude y un modelo abierto para mi TP. ¿Qué debería tener en cuenta?"}]}]
resp1 = bedrock.converse(modelId=model_id, system=system_prompt, messages=messages)
print("Turno 1:", resp1["output"]["message"]["content"][0]["text"])

# Agregamos la respuesta del modelo al historial, y el nuevo mensaje del usuario
messages.append(resp1["output"]["message"])
messages.append({"role": "user", "content": [{"text": "Y de esos criterios, ¿cuál pesa más si el corpus tiene datos sensibles?"}]})

# Turno 2: el modelo ve toda la conversación anterior
resp2 = bedrock.converse(modelId=model_id, system=system_prompt, messages=messages)
print("\nTurno 2:", resp2["output"]["message"]["content"][0]["text"])

**🏠 Practicá (tarea, después de la clase)**: sacá la línea `messages.append(resp1[...])` (no reenvíes la respuesta anterior) y hacé la misma pregunta de seguimiento ("¿cuál pesa más...?") directamente. Compará la respuesta — ¿el modelo entiende a qué te referís con "esos criterios"?

## 5. Zero-shot vs. few-shot prompting

- **Zero-shot**: se le pide la tarea al modelo sin ejemplos, confiando en que ya "sabe" hacerla por su entrenamiento.
- **Few-shot**: se incluyen 1 o más ejemplos de entrada→salida en el prompt, para fijar el formato o el criterio esperado — útil cuando la salida tiene un formato específico o el criterio es ambiguo en zero-shot.

In [ ]:
# Zero-shot: sin ejemplos
zero_shot_messages = [
    {"role": "user", "content": [{"text": "Clasificá el sentimiento de este comentario como positivo, negativo o neutro: "
                                            "'La verdad no está mal, pero esperaba más por el precio.'"}]}
]

resp_zero = bedrock.converse(modelId=model_id, messages=zero_shot_messages)
print("Zero-shot:", resp_zero["output"]["message"]["content"][0]["text"])

In [ ]:
# Few-shot: con ejemplos que fijan el formato de salida exacto
few_shot_prompt = '''Clasificá el sentimiento como POSITIVO, NEGATIVO o NEUTRO. Respondé solo con la etiqueta.

Comentario: "Excelente atención, volvería sin dudarlo."
Etiqueta: POSITIVO

Comentario: "Nunca más compro acá, un desastre."
Etiqueta: NEGATIVO

Comentario: "La verdad no está mal, pero esperaba más por el precio."
Etiqueta:'''

few_shot_messages = [{"role": "user", "content": [{"text": few_shot_prompt}]}]

resp_few = bedrock.converse(modelId=model_id, messages=few_shot_messages)
print("Few-shot:", resp_few["output"]["message"]["content"][0]["text"])

Notá la diferencia: zero-shot devuelve una respuesta en lenguaje natural más libre; few-shot, gracias a los ejemplos, tiende a devolver exactamente el formato pedido (una sola etiqueta) — esto importa mucho cuando la salida de un LLM alimenta a otro sistema (lo vamos a formalizar con *structured outputs* en la Clase 3).

**🏠 Practicá (tarea, después de la clase)**: escribí tu propio prompt few-shot (3 ejemplos) para una tarea distinta — por ejemplo, extraer de un comentario si el reclamo es sobre "envío", "producto" o "atención". Probalo primero en zero-shot para ver si ya alcanza, y después en few-shot para comparar.

In [ ]:
# Tu prompt few-shot acá
mi_prompt_few_shot = '''# TODO: escribí tus 3 ejemplos + el caso a clasificar'''

# resp_mio = bedrock.converse(modelId=model_id, messages=[{"role": "user", "content": [{"text": mi_prompt_few_shot}]}])
# print(resp_mio["output"]["message"]["content"][0]["text"])

## 6. Delimitación de instrucciones y contexto

Cuando el prompt combina instrucciones + contexto/datos + pregunta, conviene **delimitar claramente cada parte** (con XML tags, marcadores tipo `###`, o secciones con títulos). Esto reduce ambigüedad y hace más difícil que el modelo confunda una instrucción con un dato a procesar (base de la prevención de prompt injection, que se retoma en la Clase 11).

In [ ]:
prompt_delimitado = '''Sos un asistente que responde preguntas usando SOLO la información del contexto.
Si la respuesta no está en el contexto, decí explícitamente que no tenés esa información.

<contexto>
El programa tiene 12 clases semanales de 3 horas cada una, modalidad virtual.
El Trabajo Práctico Final Integrador es único para toda la cohorte: un sistema RAG
más un flujo con agentes, desplegado vía API.
</contexto>

<pregunta>
¿Cuántas clases tiene el programa y de qué modalidad son?
</pregunta>'''

resp = bedrock.converse(modelId=model_id, messages=[{"role": "user", "content": [{"text": prompt_delimitado}]}])
print(resp["output"]["message"]["content"][0]["text"])

**🏠 Practicá (tarea, después de la clase)**: agregá una segunda pregunta dentro del mismo `<pregunta>` que no se pueda responder con el contexto dado (ej. "¿cuánto cuesta el curso?"). Confirmá que el modelo diga explícitamente que no tiene esa información, en vez de inventar una respuesta — esto es la base de las "respuestas sustentadas" que vamos a formalizar en la Clase 6 (RAG).

## 7. Parámetros de generación

| Parámetro | Efecto |
|---|---|
| `temperature` | Qué tan determinística/creativa es la generación (ver Clase 1). |
| `topP` | Recorta el muestreo a los tokens más probables cuya probabilidad acumulada supera `topP`. |
| `maxTokens` | Límite de tokens de salida — corta la respuesta si se excede. |
| `stopSequences` | Cadenas que, si aparecen, terminan la generación inmediatamente. |

En Bedrock estos van en `inferenceConfig`. Parámetros específicos de un proveedor (ej. `top_k` de Anthropic) van en `additionalModelRequestFields`.

In [ ]:
inference_config = {
    "temperature": 0.2,     # bajo -> respuestas más consistentes, buena elección para tareas de extracción
    "topP": 0.9,
    "maxTokens": 200,
    "stopSequences": ["\n\n"],
}

resp = bedrock.converse(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": "Listá 3 ventajas de usar RAG en vez de fine-tuning, una por línea."}]}],
    inferenceConfig=inference_config,
)

print(resp["output"]["message"]["content"][0]["text"])
print("\nStop reason:", resp["stopReason"])

**Ejercicio mental rápido** (para discutir en clase): ¿qué `temperature` usarían para...
- Extraer un JSON de una factura escaneada → 

- Generar variantes creativas de un mensaje de marketing → 

- Resumir un documento técnico manteniendo los términos exactos → 

**🏠 Practicá (tarea, después de la clase)**: corré la misma consulta ("Listá 3 ventajas de usar RAG...") con `temperature` en 0.0, 0.7 y 1.5, dejando el resto de los parámetros igual. Corré cada una dos veces. ¿Con qué temperatura las dos corridas te dan resultados casi idénticos? ¿Con cuál varían más?

In [ ]:
for temp in [0.0, 0.7, 1.5]:
    cfg = {"temperature": temp, "maxTokens": 150}
    r = bedrock.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": "Listá 3 ventajas de usar RAG en vez de fine-tuning, una por línea."}]}],
        inferenceConfig=cfg,
    )
    print(f"--- temperature={temp} ---")
    print(r["output"]["message"]["content"][0]["text"])
    print()

## 8. Manejo básico de errores

Las llamadas a una API externa **van a fallar** en producción: rate limits, timeouts, validación de input, credenciales vencidas. Un cliente robusto no asume que todo sale bien.

In [ ]:
from botocore.exceptions import ClientError
import time


def generar_con_reintentos(bedrock_client, model_id, messages, system=None,
                            inference_config=None, max_reintentos=3):
    """Llama a Converse con reintentos y backoff exponencial ante throttling."""
    kwargs = {"modelId": model_id, "messages": messages}
    if system:
        kwargs["system"] = system
    if inference_config:
        kwargs["inferenceConfig"] = inference_config

    for intento in range(max_reintentos):
        try:
            return bedrock_client.converse(**kwargs)
        except ClientError as err:
            codigo = err.response["Error"]["Code"]
            if codigo == "ThrottlingException" and intento < max_reintentos - 1:
                espera = 2 ** intento  # backoff exponencial: 1s, 2s, 4s...
                print(f"Rate limit alcanzado, reintentando en {espera}s...")
                time.sleep(espera)
                continue
            if codigo == "ValidationException":
                raise ValueError(f"Request inválido: {err.response['Error']['Message']}") from err
            raise  # cualquier otro error de cliente, no lo tragamos silenciosamente

    raise RuntimeError("Se agotaron los reintentos por throttling.")


# Prueba: esto debería funcionar sin necesitar ningún reintento
resp = generar_con_reintentos(
    bedrock, model_id,
    messages=[{"role": "user", "content": [{"text": "Decí 'ok' y nada más."}]}],
)
print(resp["output"]["message"]["content"][0]["text"])

Errores comunes y cómo se distinguen (`err.response["Error"]["Code"]`):

- **`ThrottlingException`**: superaste el rate limit — reintentar con backoff.
- **`ValidationException`**: el request está mal formado (ej. `maxTokens` fuera de rango) — no tiene sentido reintentar sin corregirlo, hay que arreglar el input.
- **`AccessDeniedException`**: credenciales sin permiso para ese modelo — revisar `.env` y accesos habilitados.
- **Timeouts de red**: `botocore.exceptions.ConnectTimeoutError` / `ReadTimeoutError` — sí conviene reintentar.

**🏠 Practicá (tarea, después de la clase)**: provocá un `ValidationException` a propósito, mandando un `inferenceConfig` con `maxTokens` negativo (ej. `{"maxTokens": -10}`). Confirmá que `generar_con_reintentos` lance el `ValueError` con mensaje claro (no un `ClientError` crudo), y que NO haya reintentado 3 veces esperando cada vez más — un error de validación no se arregla solo.

In [ ]:
try:
    generar_con_reintentos(
        bedrock, model_id,
        messages=[{"role": "user", "content": [{"text": "Hola"}]}],
        inference_config={"maxTokens": -10},
    )
except ValueError as e:
    print("Error capturado correctamente:", e)

## 9. Síncrono vs. asíncrono

- **Síncrono**: cada llamada bloquea hasta tener respuesta. Simple, pero si hay que hacer 10 llamadas independientes, el tiempo total es la suma de las 10.
- **Asíncrono / concurrente**: se disparan varias llamadas en paralelo y se espera a que terminen todas — el tiempo total es aproximadamente el de la llamada más lenta, no la suma.

Para I/O-bound como llamadas HTTP a una API, la ganancia de correr en paralelo es grande. Boto3 no es nativamente async, así que la forma más simple es un pool de threads (`concurrent.futures`); para un cliente async "de verdad" existe `aioboto3` (no lo usamos hoy, alcanza con ver el concepto).

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

preguntas = [
    "Definí 'token' en una oración.",
    "Definí 'embedding' en una oración.",
    "Definí 'temperatura' en una oración.",
]

def preguntar(pregunta):
    resp = bedrock.converse(modelId=model_id, messages=[{"role": "user", "content": [{"text": pregunta}]}])
    return resp["output"]["message"]["content"][0]["text"]

# Secuencial
t0 = time.time()
resultados_seq = [preguntar(p) for p in preguntas]
t_seq = time.time() - t0

# Concurrente (3 llamadas en paralelo)
t0 = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    resultados_par = list(pool.map(preguntar, preguntas))
t_par = time.time() - t0

print(f"Secuencial: {t_seq:.2f}s")
print(f"Concurrente: {t_par:.2f}s")

**🏠 Practicá (tarea, después de la clase)**: subí la lista `preguntas` a 6 ítems y volvé a medir secuencial vs. concurrente (dejando `max_workers=3`). ¿El tiempo concurrente se duplica igual que el secuencial, o escala distinto? ¿Qué pasaría si subieras `max_workers` a 6?

## 10. Laboratorio: cliente básico reutilizable

**🏠 Tarea de la semana — entregar/completar antes de la Clase 3.** El docente presenta la consigna y muestra el esqueleto en vivo durante la clase, pero la resolución es trabajo individual asincrónico, no se completa durante las 3 horas de la clase.

**Checkpoint del proyecto integrador (Clase 2)**: cliente funcional para consumir el modelo generativo, construido de forma que se pueda reutilizar en las próximas clases.

### Consigna

**Importante: no hace falta que uses Bedrock.** Usá el proveedor que tengas disponible — Anthropic, OpenAI, Google o Ollama local son todos válidos. Completá la clase `LLMClient` de abajo (reemplazá los `# TODO`) para que:

1. Cargue configuración desde variables de entorno, incluyendo `self.provider = os.environ.get("LLM_PROVIDER", "bedrock")` (ya resuelto en `__init__`).
2. `generate(prompt, system=None, temperature=0.7, max_tokens=500)` llame a la función que corresponda **según `self.provider`** (`llamar_bedrock`, `llamar_anthropic`, `llamar_openai`, `llamar_google` o `llamar_ollama`, todas definidas en la sección 2) y devuelva **solo el texto** de la respuesta.
3. Agregues manejo de reintentos ante rate limiting: si usás Bedrock, reutilizá `generar_con_reintentos` de la sección 8 tal cual; si usás otro proveedor, aplicá el mismo patrón (backoff exponencial) capturando la excepción de rate limit específica de tu SDK (`openai.RateLimitError` para OpenAI, `anthropic.RateLimitError` para Anthropic directo — para Google/Ollama alcanza con un `try/except Exception` genérico con backoff, no hace falta la excepción exacta).
4. Que un error de validación (ej. `max_tokens` inválido) se relance con un mensaje claro, no un traceback críptico del SDK.
5. (Opcional) Agregues un método `generate_batch(prompts)` que resuelva una lista de prompts de forma concurrente, usando el patrón de la sección 9.

### Criterio de éxito

- `LLMClient().generate("¿Cuál es la capital de Francia?")` devuelve un string con la respuesta, sin lanzar excepción, **con el proveedor que hayas configurado en tu `.env`**.
- Si se llama con un `max_tokens` inválido (ej. negativo), se obtiene un error claro y explicable, no un traceback críptico del SDK.
- El cliente no tiene ninguna credencial hardcodeada — todo viene de `os.environ` / `.env`.
- Si le mostrás tu código a un compañero que usa un proveedor distinto al tuyo, tiene que poder cambiar `LLM_PROVIDER` en su `.env` y que funcione sin tocar tu código (más allá de tener las funciones `llamar_*` de los otros proveedores disponibles).

In [ ]:
class LLMClient:
    def __init__(self):
        load_dotenv()
        self.provider = os.environ.get("LLM_PROVIDER", "bedrock")

    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 500) -> str:
        # TODO: según self.provider, llamar a la función llamar_bedrock / llamar_anthropic /
        # llamar_openai / llamar_google / llamar_ollama correspondiente (sección 2),
        # envuelta en manejo de reintentos/errores (ver consigna, puntos 3 y 4).
        raise NotImplementedError("Completar generate()")

    def generate_batch(self, prompts: list[str]) -> list[str]:
        # TODO (opcional): resolver la lista de prompts de forma concurrente
        raise NotImplementedError("Completar generate_batch() (opcional)")


# Prueba rápida (va a fallar hasta que completes los TODOs de arriba):
# client = LLMClient()
# print(client.generate("¿Cuál es la capital de Francia?"))

## Cierre

- Hoy pasamos de "entender el modelo" a **programar contra él**: entorno, credenciales, primera llamada, prompting básico, parámetros y manejo de errores.
- El cliente de hoy soporta varios proveedores mediante un `if/elif` sobre `self.provider` — funciona, pero no escala bien (¿y si mañana aparece un proveedor nuevo?). En la Clase 3 lo vamos a poner detrás de una interfaz propia más prolija, y vamos a agregar salidas estructuradas (Pydantic) en vez de texto libre.
- Antes de la próxima clase: dejar el `LLMClient` de la sección 10 funcionando **con tu proveedor** — la Clase 3 arranca extendiéndolo.